# Create one earnings dossier per ticker

Given a list of ticker symbols, query Yahoo Finance through `yfinance` and write `knowledge/dossier/{TICKER}.yaml`. Tickers are normalized to uppercase and used as canonical identifiers. Each dossier retains only the eight most recent historical earnings results.

Timing policy: BMO uses that trading day's close-to-close return; AMC uses the next trading day's return; unavailable or ambiguous timing is marked `unknown` and its reaction values remain null. Historical guidance is omitted when unavailable because yfinance has no dependable historical company-guidance endpoint.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Callable, Sequence
import time
import warnings

import numpy as np
import pandas as pd
import yaml
import yfinance as yf
from tqdm.auto import tqdm

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'knowledge').is_dir()), Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'knowledge' / 'dossier'
INDUSTRY_MAP_PATH = PROJECT_ROOT / 'knowledge' / 'mappings' / 'industry_map.csv'
SURPRISE_SOURCE = 'Yahoo Finance (via yfinance)'


In [ ]:
def canonical_ticker(value: Any) -> str:
    ticker = str(value).strip().upper()
    if not ticker or ticker == 'NAN':
        raise ValueError('Ticker cannot be empty')
    if any(char in ticker for char in ('/', '\\')) or ticker in {'.', '..'}:
        raise ValueError(f'Unsafe ticker identifier: {ticker!r}')
    return ticker


def scalar(value: Any, digits: int = 4):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return round(float(value), digits)
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    return value


def empty_prices() -> pd.Series:
    return pd.Series(dtype=float, index=pd.DatetimeIndex([], name='Date'))


def fetch_adjusted_prices(symbol: str, period: str = '10y', raise_errors: bool = False) -> pd.Series:
    try:
        instrument = yf.Ticker(symbol)
        frame = instrument.history(period=period, interval='1d', auto_adjust=False, actions=False, repair=False)
        column = 'Adj Close'
        if frame is None or frame.empty or column not in frame.columns:
            frame = instrument.history(period=period, interval='1d', auto_adjust=True, actions=False, repair=False)
            column = 'Close'
        if frame is None or frame.empty or column not in frame.columns:
            return empty_prices()
        index = pd.to_datetime(frame.index, errors='coerce', utc=True).tz_convert(None).normalize()
        prices = pd.Series(pd.to_numeric(frame[column], errors='coerce').to_numpy(), index=index)
        prices = prices[~prices.index.isna() & prices.notna()]
        return prices.groupby(level=0).last().sort_index().astype(float)
    except Exception as exc:
        if raise_errors:
            raise
        warnings.warn(f'{symbol}: adjusted prices unavailable: {exc}')
        return empty_prices()


def classify_earnings_time(timestamp: Any) -> str:
    if timestamp is None or pd.isna(timestamp):
        return 'unknown'
    try:
        ts = pd.Timestamp(timestamp)
        if ts.tzinfo is not None:
            ts = ts.tz_convert('America/New_York')
        minutes = ts.hour * 60 + ts.minute
    except (TypeError, ValueError):
        return 'unknown'
    if minutes == 0:
        return 'unknown'
    if minutes <= 9 * 60 + 30:
        return 'before_market_open'
    if minutes >= 16 * 60:
        return 'after_market_close'
    return 'unknown'


def fetch_earnings_events(symbol: str, limit: int = 8, raise_errors: bool = False) -> pd.DataFrame:
    columns = ['ticker', 'fiscal_quarter', 'announcement_date', 'timing', 'earnings_surprise_pct', 'surprise_source']
    try:
        limit = max(1, min(int(limit), 100))
        raw = yf.Ticker(symbol).get_earnings_dates(limit=limit)
        if raw is None or raw.empty:
            return pd.DataFrame(columns=columns)
        surprise_column = next((c for c in ('Surprise(%)', 'Surprise (%)', 'surprisePercent') if c in raw.columns), None)
        today = pd.Timestamp.now(tz='America/New_York').date()
        rows = []
        for position, timestamp in enumerate(raw.index):
            ts = pd.Timestamp(timestamp)
            local_ts = ts.tz_convert('America/New_York') if ts.tzinfo is not None else ts
            if local_ts.date() > today:
                continue
            source_row = raw.iloc[position]
            surprise = pd.to_numeric(source_row.get(surprise_column), errors='coerce') if surprise_column else np.nan
            rows.append({
                'ticker': symbol,
                'fiscal_quarter': None,
                'announcement_date': local_ts.normalize().tz_localize(None),
                'timing': classify_earnings_time(ts),
                'earnings_surprise_pct': scalar(surprise),
                'surprise_source': SURPRISE_SOURCE if pd.notna(surprise) else None,
            })
        events = pd.DataFrame(rows, columns=columns)
        return events.sort_values('announcement_date', ascending=False).head(limit).sort_values('announcement_date')
    except Exception as exc:
        if raise_errors:
            raise
        warnings.warn(f'{symbol}: earnings history unavailable: {exc}')
        return pd.DataFrame(columns=columns)


def fetch_historical_guidance(symbol: str) -> pd.DataFrame:
    # Provider boundary: yfinance does not expose reliable historical guidance.
    return pd.DataFrame()


def call_with_retries(call: Callable[[], Any], label: str, attempts: int = 5, delay_seconds: float = 3.0):
    if attempts < 1:
        raise ValueError('attempts must be at least 1')
    for attempt in range(1, attempts + 1):
        try:
            return call()
        except Exception as exc:
            if attempt == attempts:
                raise
            warnings.warn(f'{label} failed on attempt {attempt}/{attempts}: {exc}; retrying in {delay_seconds:g}s')
            time.sleep(delay_seconds)


In [ ]:
def reaction_session(announcement_date: Any, timing: str, sessions: pd.DatetimeIndex):
    if timing == 'unknown' or pd.isna(announcement_date) or sessions.empty:
        return None
    date = pd.Timestamp(announcement_date).normalize()
    side = 'left' if timing == 'before_market_open' else 'right'
    position = sessions.searchsorted(date, side=side)
    return sessions[position] if position < len(sessions) else None


def reaction_statistics(reactions: list[dict[str, Any]]) -> dict[str, Any]:
    usable = [row for row in reactions if row.get('abnormal_return_pct') is not None]
    values = np.array([row['abnormal_return_pct'] for row in usable], dtype=float)
    beats = np.array([row['abnormal_return_pct'] for row in usable if row.get('earnings_surprise_pct') is not None and row['earnings_surprise_pct'] > 0], dtype=float)
    misses = np.array([row['abnormal_return_pct'] for row in usable if row.get('earnings_surprise_pct') is not None and row['earnings_surprise_pct'] < 0], dtype=float)

    def stat(array: np.ndarray, function):
        return round(float(function(array)), 4) if array.size else None

    return {
        'observations': int(values.size),
        'median_abnormal_return_pct': stat(values, np.median),
        'mean_abnormal_return_pct': stat(values, np.mean),
        'positive_reaction_rate': stat(values, lambda x: np.mean(x > 0)),
        'median_absolute_reaction_pct': stat(values, lambda x: np.median(np.abs(x))),
        'beat_observations': int(beats.size),
        'median_reaction_after_beat_pct': stat(beats, np.median),
        'miss_observations': int(misses.size),
        'median_reaction_after_miss_pct': stat(misses, np.median),
    }


def build_dossier(ticker: str, stock_prices: pd.Series, vti_prices: pd.Series, earnings: pd.DataFrame) -> dict[str, Any]:
    ticker = canonical_ticker(ticker)
    sessions = stock_prices.index.intersection(vti_prices.index).sort_values()
    stock_returns = stock_prices.reindex(sessions).pct_change(fill_method=None) * 100
    vti_returns = vti_prices.reindex(sessions).pct_change(fill_method=None) * 100
    reactions = []
    for _, event in earnings.sort_values('announcement_date').iterrows():
        timing = event.get('timing', 'unknown')
        session = reaction_session(event.get('announcement_date'), timing, sessions)
        stock_return = stock_returns.get(session) if session is not None else None
        vti_return = vti_returns.get(session) if session is not None else None
        if stock_return is None or vti_return is None or pd.isna(stock_return) or pd.isna(vti_return):
            stock_return = vti_return = abnormal_return = None
        else:
            abnormal_return = float(stock_return - vti_return)
        reactions.append({
            'fiscal_quarter': scalar(event.get('fiscal_quarter')),
            'timing': timing,
            'reaction_date': scalar(session),
            'stock_return_pct': scalar(stock_return),
            'vti_return_pct': scalar(vti_return),
            'abnormal_return_pct': scalar(abnormal_return),
            'earnings_surprise_pct': scalar(event.get('earnings_surprise_pct')),
            'surprise_source': scalar(event.get('surprise_source')),
        })
    return {'ticker': ticker, 'prior_reactions': reactions, 'reaction_statistics': reaction_statistics(reactions)}


def create_dossiers(
    tickers: Sequence[str],
    period: str = '10y',
    earnings_limit: int = 8,
    output_dir: str | Path = OUTPUT_DIR,
    skip_existing: bool = True,
    retry_attempts: int = 5,
    retry_delay_seconds: float = 3.0,
) -> dict[str, dict[str, Any]]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    try:
        vti_prices = call_with_retries(
            lambda: fetch_adjusted_prices('VTI', period=period, raise_errors=True),
            'VTI prices', retry_attempts, retry_delay_seconds,
        )
    except Exception as exc:
        warnings.warn(f'VTI prices unavailable after {retry_attempts} attempts: {exc}')
        vti_prices = empty_prices()
    for raw_ticker in tqdm(tickers, desc='Creating dossiers', unit='ticker'):
        try:
            ticker = canonical_ticker(raw_ticker)
        except Exception as exc:
            results[str(raw_ticker)] = {'path': None, 'error': str(exc)}
            continue
        destination = output_dir / f'{ticker}.yaml'
        if skip_existing and destination.exists():
            results[ticker] = {'path': str(destination), 'status': 'skipped_existing', 'warnings': []}
            continue
        warnings_for_ticker = []
        try:
            stock_prices = call_with_retries(
                lambda: fetch_adjusted_prices(ticker, period=period, raise_errors=True),
                f'{ticker} prices', retry_attempts, retry_delay_seconds,
            )
        except Exception as exc:
            warnings_for_ticker.append(f'adjusted stock prices unavailable after retries: {exc}')
            stock_prices = empty_prices()
        try:
            earnings = call_with_retries(
                lambda: fetch_earnings_events(ticker, limit=earnings_limit, raise_errors=True),
                f'{ticker} earnings', retry_attempts, retry_delay_seconds,
            )
        except Exception as exc:
            warnings_for_ticker.append(f'historical earnings events unavailable after retries: {exc}')
            earnings = pd.DataFrame(columns=['announcement_date'])
        _ = fetch_historical_guidance(ticker)
        if stock_prices.empty:
            warnings_for_ticker.append('adjusted stock prices unavailable')
        if vti_prices.empty:
            warnings_for_ticker.append('adjusted VTI prices unavailable')
        if earnings.empty:
            warnings_for_ticker.append('historical earnings events unavailable')
        try:
            dossier = build_dossier(ticker, stock_prices, vti_prices, earnings)
        except Exception as exc:
            warnings_for_ticker.append(f'dossier calculation failed: {exc}')
            dossier = {'ticker': ticker, 'prior_reactions': [], 'reaction_statistics': reaction_statistics([])}
        with destination.open('w', encoding='utf-8') as handle:
            yaml.safe_dump(dossier, handle, sort_keys=False, allow_unicode=True)
        results[ticker] = {'path': str(destination), 'status': 'written', 'warnings': warnings_for_ticker}
    return results


## Configure and run

Read `knowledge/mappings/industry_map.csv`, extract and deduplicate its `ticker` column, then run the resumable batch. Existing files in `knowledge/dossier` are skipped.

In [ ]:
industry_map = pd.read_csv(INDUSTRY_MAP_PATH)
if 'ticker' not in industry_map.columns:
    raise ValueError(f'{INDUSTRY_MAP_PATH} must contain a ticker column')

TICKERS = list(dict.fromkeys(
    canonical_ticker(value) for value in industry_map['ticker'].dropna() if str(value).strip()
))
print(f'Loaded {len(TICKERS):,} unique tickers from {INDUSTRY_MAP_PATH}')

results = create_dossiers(
    TICKERS,
    output_dir=OUTPUT_DIR,
    earnings_limit=8,
    skip_existing=True,
    retry_attempts=5,
    retry_delay_seconds=3,
)
results
